In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import base64
from io import BytesIO
from PIL import Image
import gradio as gr

GPT_IMAGE_MODEL = "dall-e-3"
GPT_MODEL = "gpt-4.1-mini"

In [ ]:
load_dotenv()


In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    print("Please set your OPENAI_API_KEY environment variable.")
else:
    print("API key loaded successfully.")

openai = OpenAI(api_key=api_key)

In [ ]:
def visualize_therapy(prompt):
    response_image = openai.images.generate(
        prompt=prompt,
        model=GPT_IMAGE_MODEL,
        size="1024x1024",
        n=1,
        response_format="b64_json"
    )
    image_b64 = response_image.data[0].b64_json
    image_data = base64.b64decode(image_b64)
    return Image.open(BytesIO(image_data))

def vocalize_therapy(prompt):
    response_audio = openai.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=prompt  # 'input' parameter, not 'prompt'
    )
    # The response is audio bytes, not base64
    return response_audio.content

In [ ]:
visualize_therapy_tool = {
    "type": "function",
    "function": {
        "name": "visualize_therapy",
        "description": "Generates a therapeutic image based on the user's feelings or situation.",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "A description of the user's feelings or situation to generate a relevant therapeutic image."
                }
            },
            "required": ["prompt"],
            "additionalProperties": False
        }
    }
}

vocalize_therapy_tool = {
    "type": "function",
    "function": {
        "name": "vocalize_therapy",
        "description": "Advises the user with their feelings or situation through generating a therapy voice",
        "parameters": {
            "type": "object",
            "properties": {
                "prompt": {
                    "type": "string",
                    "description": "An advice to help the user to resolve their emotional or mental issue and improve their wellbeing."
                }
            },
            "required": ["prompt"],
            "additionalProperties": False
        }
    }
}

system_message = """
    You're a therapy assistant who helps the user to resolve their emotional or mental
      issues by advising the user through audio and showing a helpful image that is 
      related to their situation. 
"""

tools = [
    {"definition": visualize_therapy_tool, "function": visualize_therapy},
    {"definition": vocalize_therapy_tool, "function": vocalize_therapy}
]

In [ ]:
def chat_with_therapy_assistant(history):
    messages = [{"role": "system", "content": system_message}] + history
    
    response = openai.chat.completions.create(
        model=GPT_MODEL,
        messages=messages,
        tools=[tool["definition"] for tool in tools]
    )
    
    message = response.choices[0].message
    image, audio = None, None
    
    if response.choices[0].finish_reason == "tool_calls":
        tool_calls = response.choices[0].message.tool_calls
        image, audio = handle_tool_calls(tool_calls)
        
    response_text = message.content if message.content else "Here's something to help you feel better."
    return image, audio, history + [{"role": "assistant", "content": response_text}]

def handle_tool_calls(tool_calls):
    image, audio = None, None
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_args = eval(tool_call.function.arguments)
        
        for tool in tools:
            if tool["definition"]["function"]["name"] == tool_name:
                if tool_name == "visualize_therapy":
                    image = tool["function"](**tool_args)
                elif tool_name == "vocalize_therapy":
                    audio = tool["function"](**tool_args)
    return image, audio

In [ ]:
def put_chat_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=400, type="messages")
        image_output = gr.Image(height=400, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(label="Therapeutic Audio", interactive=False, autoplay=True)
    with gr.Row():
        message = gr.Textbox(placeholder="Describe your feelings or situation...", label="Your Message")

    message.submit(
        put_chat_in_chatbot, 
        inputs=[message, chatbot], 
        outputs=[message, chatbot]
    ).then(
        chat_with_therapy_assistant,
        inputs=chatbot,
        outputs=[image_output, audio_output, chatbot]
    )

ui.launch(inbrowser=True)